# Activity 12.6: Developing Python skills with the wildifires-and-blooms lab data

<img src="https://i0.wp.com/datalab.marine.rutgers.edu/wp-content/uploads/2025/06/New-X.0.3.png?ssl=1" width="450px" align="right">

The data you have been using in this lab was collected by two  moorings of the Coastal Endurance array:
*   The Oregon shelf surface mooring (activity 12.3)
*   The Oregon offshore surface mooring (activities 12.4-12.6)

In this Python activity, you'll work with data from all three surface moorings of the Coastal Endurance Oregon Line. Before you work on this activity, you are strongly encouraged to visit:

*   The Python Quick Intro activity "[Pioneer MAB Storm Example Activity / Python Quickstart](https://colab.research.google.com/github/ooi-data-lab/data-labs-2024/blob/main/Pioneer%20MAB%20Storm%20-%20Quick%20Start.ipynb)" on the [OOI Data Python Notebooks page](https://datalab.marine.rutgers.edu/python-notebooks/),
*   The activities of the wildfires lab, expecially the ones mentioned above.

By the end of this activity, you will be able to:
*   Navigate the [ERDDAP (the Environmental Research Division's Data Access Program) server](https://erddap.dataexplorer.oceanobservatories.org/erddap/index.html) and use it to load data into a programming environment, like Google's Colab. A more interactive way to engage with the data can be found on [this page](https://dataexplorer.oceanobservatories.org/).
*   Produce time-series graphs like the ones you used in activities 12.3-12.6, including atmospheric and surface-ocean parameters, using Python
*   Adjust graph parameters, including colors, chemical formulas, units, axis scales, etc.
*   Overlay multiple data sets on the same plot or stack graphs to generate multipanel figures
*   Use your graphs to answer questions on the path of the ash plume and its impact on the water column

# 1 Data gathering

## 1.1 Notebook setup

First, we must set up this Colab notebook so it can run code and process the necessary libraries.

In [ ]:
# Install erddapy
!pip install erddapy

In [ ]:
from erddapy import ERDDAP
import pandas as pd
import matplotlib.pyplot as plt

## 1.2 Links to data

We will now connect with the OOI data library, and download data from the two moorings.

1.2a Connect with the server and define the format of the files we will access (csv).

In [ ]:
# Setup Erddap connection
server = "https://erddap.dataexplorer.oceanobservatories.org/erddap/"
e = ERDDAP(
    server=server,
    protocol="tabledap",
    response="csv",
)

1.2b With this next command, you can access meteorological data from the Oregon Shelf Surface Mooring. Note that the data of all the parameters we are interested in have the same ID (ooi-ce02shsm-sbd11-06-metbka000) so we only need one command. We'll call this dataset BulkMeteoShelf.

In [ ]:
url_BulkMeteoShelf = e.get_download_url(
    dataset_id = "ooi-ce02shsm-sbd11-06-metbka000",
    constraints = {
      "time>=": "2020-09-06",
      "time<=": "2020-09-21",
    }
)

1.2c Now, use the example above and access sea-surface data from the Oregon Offshore Surface Mooring for the same time period. Note that the parameters we are interested in have different IDs, therefore we need multiple commands, one for each ID or instrument.

Use the command above as an example and modify it for each of the datasets you'll need, by replacing their names and dataset IDs.

This is a list of the dataset IDs and suggestions on what to call each one:

*   ooi-ce04ossm-rid27-02-flortd000 - 3WavelengthFluorometerOffshore
*   ooi-ce04ossm-rid27-04-dostad000 - O2Offshore
*   ooi-ce04ossm-rid26-07-nutnrb000 - NitrateOffshore

In [ ]:
# Enter your code here

<details>
  <summary>Click here to reveal code that may work!</summary>

```python
# Instructor's resource: 1.2c

url_3WavelengthFluorometerOffshore = e.get_download_url(
    dataset_id = "ooi-ce04ossm-rid27-02-flortd000",
    constraints = {
      "time>=": "2020-09-06",
      "time<=": "2020-09-21",
    }
)
url_O2Offshore = e.get_download_url(
    dataset_id = "ooi-ce04ossm-rid27-04-dostad000",
    constraints = {
      "time>=": "2020-09-06",
      "time<=": "2020-09-21",
    }
)
url_NitrateOffshore = e.get_download_url(
    dataset_id = "ooi-ce04ossm-rid26-07-nutnrb000",
    constraints = {
      "time>=": "2020-09-06",
      "time<=": "2020-09-21",
    }
)
```

</details>

## 1.3 Data setup

We will now load and clean the data we want in our notebook from the data links we established in the previous step.

1.3a Load data from the two moorings

First, load the data from the Oregon Shelf Mooring's meteorological package using the command that follows.

In [ ]:
df_BulkMeteoShelf = pd.read_csv(url_BulkMeteoShelf, index_col='time', parse_dates=True, skiprows=[1])

Now, use it as an example to load the data from the Oregon Offshore mooring, by copying the code above and using the name of each dataset you used in 1.2c. These were the recommended names, but make sure you use your own if you changed them:

*   3WavelengthFluorometerOffshore
*   O2Offshore
*   NitrateOffshore

In [ ]:
# Enter your code here

<details>
  <summary>Click here to reveal code that may work!</summary>

```python
# Instructor's resource: 1.3a

df_3WavelengthFluorometerOffshore = pd.read_csv(url_3WavelengthFluorometerOffshore, index_col='time', parse_dates=True, skiprows=[1])
df_O2Offshore = pd.read_csv(url_O2Offshore, index_col='time', parse_dates=True, skiprows=[1])
df_NitrateOffshore = pd.read_csv(url_NitrateOffshore, index_col='time', parse_dates=True, skiprows=[1])
```

</details>

1.3b At this point, we can finally take a look at the data and spot the parameters we are actually interested in. Before we do so, we will make sure that we can see all the columns of each dataset by running the following command.

In [ ]:
pd.set_option('display.max_columns', None)

The code that follows shows the first few lines of the meteorological dataset from the Oregon Shelf Mooring. Focus on the headers! Specifically, look for the ones that match the parameters you worked with in activity 12.3:

*   Air temperature
*   Relative humidity
*   Eastward wind speed
*   Northward wind speed
*   Net shortwave irradiance

Note the exact syntax of each header. We'll need it in part c of this step of the Python activity.

In [ ]:
df_BulkMeteoShelf.head()

Use the syntax of this command, generate commands to view the other datasets we've called already (make sure you use the correct names, if you didn't use the suggested ones shown below). The fluorometer (3WavelengthFluorometerOffshore) records the following key parameters:

*   Sea surface temperature
*   Optical backscatter
*   Fluorometric CDOM
*   Chlorophyll *a*

Surely, you've noticed how some header names are precise and possibly long, whereas others are very short and creative. Happy hunting!

The other two datasets we are interested in contain the mole concentrations of the two chemicals:

*   Oxygen (O2Offshore)
*   Nitrate sensor (NitrateOffshore)



In [ ]:
# Enter your code here

<details>
  <summary>Click here to reveal code that may work!</summary>

```python
# Instructor's resource: 1.3b
# Place the three segments below in separate code boxes

# Fluorometer Offshore
df_3WavelengthFluorometerOffshore.head()
# Oxygen Offshore
df_O2Offshore.head()
# Nitrate Offshore
df_NitrateOffshore.head()
```

</details>

1.3c Now, we will define the data of each parameter and, while doing so, we will "clean" it by removing rows with missing data in the same command. Yes, data can be missing because instruments malfunction or aren't working altogether, and this can mess up our plots.

As an example, below you can see the command defining the air temperature (header title: air_temperature) at the Oregon Shelf mooring (BulkMeteoShelf) and it's now called "AirTemperatureShelf".

In [ ]:
df_AirTemperatureShelf = df_BulkMeteoShelf.dropna(subset='air_temperature')

Now, write commands for all other parameters you encountered in this lab activity. You will need the following:

*   The name of the dataset, e.g., BulkMeteoShelf
*   The name of the parameter exactly as identified in the header, e.g., air_temperature
*   A name you can give the parameter dataset, e.g. AirTemperatureShelf. This name includes information on both the parameter and the mooring of the array, which can be useful if you have data of the same parameter from various locations.

Here's a list of the parameters for your convenience, as well as suggested names for each parameter dataset:

*   Relative humidity, RelativeHumidityShelf
*   Eastward wind speed, EastwardWindShelf
*   Northward wind speed, NorthwardWindShelf
*   Net shortwave irradiance, ShortWaveFluxShelf
*   Sea surface temperature, SeaSurfaceTemperatureOffshore
*   Optical backscatter, OpticalBackscatterOffshore
*   Fluorometric CDOM, CDOMOffshore
*   Chlorophyll *a*, ChlaOffshore
*   Oxygen, O2Offshore
*   Nitrate, NitrateOffshore

In [ ]:
# Enter your code here

<details>
  <summary>Click here to reveal code that may work!</summary>

```python
# Instructor's resource: 1.3c
df_RelativeHumidityShelf = df_BulkMeteoShelf.dropna(subset='relative_humidity')
df_EastwardWindShelf = df_BulkMeteoShelf.dropna(subset='eastward_wind')
df_NorthwardWindShelf = df_BulkMeteoShelf.dropna(subset='northward_wind')
df_ShortWaveFluxShelf = df_BulkMeteoShelf.dropna(subset='downwelling_shortwave_flux_in_air')
df_SeaSurfaceTemperatureOffshore = df_3WavelengthFluorometerOffshore.dropna(subset='sea_water_temperature')
df_OpticalBackscatterOffshore = df_3WavelengthFluorometerOffshore.dropna(subset='flubsct')
df_CDOMOffshore = df_3WavelengthFluorometerOffshore.dropna(subset='cdomflo')
df_ChlaOffshore = df_3WavelengthFluorometerOffshore.dropna(subset='mass_concentration_of_chlorophyll_a_in_sea_water')
df_O2Offshore = df_O2Offshore.dropna(subset='mole_concentration_of_dissolved_molecular_oxygen_in_sea_water')
df_NitrateOffshore = df_NitrateOffshore.dropna(subset='mole_concentration_of_nitrate_in_sea_water_suna')**bold text**
```

</details>

# 2 Time-series plots

Now that you have set up your data, generate the basic time-series plots (with time on the x-axis) that you interacted with earlier in this lab (Activities 12.3-12.6).

## 2.1 Meteorological data

Start with the meteorological data that indicate when the ash plume arrives (Activity 12.3):

*   Air temperature
*   Relative humidity
*   Eastward wind
*   Northward wind
*   Short-wave irradiance flux

Make sure you label each y-axis with the name of the plotted parameter and its units. Do the following if you need to place text in:

*   Superscript: the text should follow "^" and be enveloped in dollar signs
*   Subscript: as above, but use "_" instead of "^"

The code for the air temperature graph follows as an example.

In [ ]:
# Plot air temperature
df_AirTemperatureShelf['air_temperature'].plot(color='black')
plt.ylabel('Air temperature ($^o$C)');

In [ ]:
# Enter your code here

<details>
  <summary>Click here to reveal code that may work!</summary>

```python
# Instructor's resource: 2.1
# Place the four segments below in separate code boxes

# Plot relative humidity
df_RelativeHumidityShelf['relative_humidity'].plot(color='black')
plt.ylabel('Relative humidity (%)');

# Plot eastward wind
df_EastwardWindShelf['eastward_wind'].plot(color='black')
plt.ylabel('Eastward wind (m/s)');

# Plot northward wind
df_NorthwardWindShelf['northward_wind'].plot(color='black')
plt.ylabel('Northward wind (m/s)');

# Plot shortwave irradiance flux
df_ShortWaveFluxShelf['downwelling_shortwave_flux_in_air'].plot(color='black')
plt.ylabel('Shortwave irradiance flux (W/m$^2$)');
```

</details>

## 2.2 Ash deposition

Continue with the sea-surface data that show the deposition of the ash plume on the surface of the ocean (Activity 12.4).

Begin experimenting with graph properties like the color of the data:

*   Sea-surface temperature: black
*   Backscatter: grey
*   CDOM: orange

In [ ]:
# Enter your code here

<details>
  <summary>Click here to reveal code that may work!</summary>

```python
# Instructor's resource: 2.2
# Place the three segments below in separate code boxes

# Plot sea surface temperature
df_SeaSurfaceTemperatureOffshore['sea_water_temperature'].plot(color='black')
plt.ylabel('Sea surface temperature ($^o$C)');

# Instructor's resource: 2.2 Plot optical backscatter
df_OpticalBackscatterOffshore['flubsct'].plot(color='grey')
plt.ylabel('Optical backscatter (1/m)');

# Instructor's resource: 2.2 Plot fluorescent CDOM
df_CDOMOffshore['cdomflo'].plot(color='orange')
plt.ylabel('Fluorescent CDOM (ppb)');
```

</details>

## 2.3 Algal bloom

Finally, generate the plots that could show whether a bloom occurs as a response to the arrival of the ash plume (Activity 3).

Make sure you render the units and names in the y-axis label properly:

*   If you use the chemical formula of oxygen gas, O$_2$, make sure you place "2" in subscript by preceding it with an underscore and following it with a dollar sign,
*   Similarly, render the formula of nitrate properly with the "3" in subscript and the "-" in superscript
*   Use the Greek "m", μ, for "micro" that can be copied directly from a word processor (using the "Symbol" font style) or from this line,
*   Italicize the "a" in "chlorophyll *a*" by placing it between dollar signs.

Continue experimenting with graph properties like the color of the data:
*   Chlorophyll *a*: green
*   Oxygen: red
*   Nitrate: purple





In [ ]:
# Enter your code here

<details>
  <summary>Click here to reveal code that may work!</summary>

```python
# Instructor's resource: 2.3
# Place the three segments below in separate code boxes

# Plot chlorophyll a
df_ChlaOffshore['mass_concentration_of_chlorophyll_a_in_sea_water'].plot(color='green')
plt.ylabel('[Chlorophyll $a$] (μg/L)');

# Plot oxygen concentration
df_O2Offshore['mole_concentration_of_dissolved_molecular_oxygen_in_sea_water'].plot(color='red')
plt.ylabel('[$O_2$] (μmol/L)');

# Plot nitrate concentration with time
df_NitrateOffshore['mole_concentration_of_nitrate_in_sea_water_suna'].plot(color='purple')
plt.xlabel('')
plt.ylabel('[NO$_3^{-}$] (μmol/L)');
```

</details>

The nitrate plot poses an interesting problem, which you surely noticed: The sudden drops in the concentration values that look like straight lines. By filtering those "errant" low values, we can produce a nitrate time-series graph that is clearer.

Examine the graph and how low those straight lines extend. Which nitrate concentration value would you choose as a "filter" so that all outliers (and the straight lines) would be excluded?

You can store a new "filtered" nitrate parameter and use your preferred value after the symbol ">" to exclude any original values below it. Here's code that can exclude values below 6 μmol/L:

```python
df_NitrateOffshore_filtered = df_NitrateOffshore[df_NitrateOffshore['mole_concentration_of_nitrate_in_sea_water_suna'] > 6]
df_NitrateOffshore_filtered['mole_concentration_of_nitrate_in_sea_water_suna'].plot(color='purple')
```

Run this code and then try to plot the filtered series and see if it worked!

In [ ]:
# Enter your code here

<details>
  <summary>Click here to reveal code that may work!</summary>

```python
# Instructor's resource: 2.3 Filter extreme lows and plot nitrate concentration with time
df_NitrateOffshore_filtered = df_NitrateOffshore[df_NitrateOffshore['mole_concentration_of_nitrate_in_sea_water_suna'] > 6]
df_NitrateOffshore_filtered['mole_concentration_of_nitrate_in_sea_water_suna'].plot(color='purple')
plt.xlabel('')
plt.ylabel('[NO$_3^{-}$] (μmol/L)');
```

</details>

# 3 Data exploration

<img src="https://i0.wp.com/datalab.marine.rutgers.edu/wp-content/uploads/2025/06/New-X.0.3.png?ssl=1" width="400px" align="right">

In this lab chapter, we used atmospheric data from the Oregon shelf mooring (activity 12.3) and the Oregon offshore mooring (activities 12.4-12.6). They both actually carry instruments that monitor the atmosphere and sea-surface. Moreover, there is actually a third mooring, the Oregon inshore mooring, with sea-surface instruments. All three together constitute the so-called Oregon line.

Explore patterns in the atmospheric and sea-surface data of the Oregon line that will deepen your understanding of the effect of wildfires on sea-surface processes, and in the process build on the Python skills you practiced above and develop new ones.

## 3.1 Ash on the go!

***Question: Can we see ash get transported from land to the ocean?***

The offshore mooring actually has a meteorological sensor package (the inshore mooring doesn't). Go to the [ERDDAP (the Environmental Research Division's Data Access Program) server](https://erddap.dataexplorer.oceanobservatories.org/erddap/index.html), find its ID, and use the steps we went over in section 1 above to set up and clean the data.

To answer the question, you can compare the atmospheric parameter time-series from the shelf and offshore locations closely by comparing the timing of changes in any two or more properties above. You can do so by:

*   Stacking time-series graphs in a multipanel figure
*   Plotting two or more parameters in the same time-series graph

See the relevant examples in "[Pioneer MAB Storm Example Activity / Python Quickstart](https://colab.research.google.com/github/ooi-data-lab/data-labs-2024/blob/main/Pioneer%20MAB%20Storm%20-%20Quick%20Start.ipynb)" and feel free to paste and modify the relevant code here.

In [ ]:
# Enter your code here

<details>
  <summary>Click here to reveal code that may work!</summary>

```python
# Instructor's resource: 3.1 - Load and compare meteorological data from the shelf and offshore moorings

# Access meteorological data from offshore mooring
url_BulkMeteoOffshore = e.get_download_url(
    dataset_id = "ooi-ce04ossm-sbd11-06-metbka000",
    constraints = {
      "time>=": "2020-09-06",
      "time<=": "2020-09-21",
    }
)

# Load the data
df_BulkMeteoOffshore = pd.read_csv(url_BulkMeteoOffshore, index_col='time', parse_dates=True, skiprows=[1])

# Remove rows with NaN values
df_AirTemperatureOffshore = df_BulkMeteoOffshore.dropna(subset='air_temperature')
df_RelativeHumidityOffshore = df_BulkMeteoOffshore.dropna(subset='relative_humidity')
df_EastwardWindOffshore = df_BulkMeteoOffshore.dropna(subset='eastward_wind')
df_NorthwardWindOffshore = df_BulkMeteoOffshore.dropna(subset='northward_wind')
df_ShortWaveFluxOffshore = df_BulkMeteoOffshore.dropna(subset='downwelling_shortwave_flux_in_air')

# Plot meteorological data
fig,(ax1,ax2,ax3,ax4,ax5) = plt.subplots(5,1, sharex=True, figsize=(15,15))

df_AirTemperatureShelf['air_temperature'].plot(ax=ax1, label='Shelf mooring', color='blue')
df_AirTemperatureOffshore['air_temperature'].plot(ax=ax1, label='Offshore mooring', color='cyan')
ax1.set_xlabel('')
ax1.set_ylabel('Air temperature ($^o$C)')
ax1.legend()

df_EastwardWindShelf['eastward_wind'].plot(ax=ax3, label='Shelf mooring', color='blue')
df_EastwardWindOffshore['eastward_wind'].plot(ax=ax3, label='Offshore mooring', color='cyan')
ax3.set_xlabel('')
ax3.set_ylabel('Eastward wind speed (m/s)')
ax3.legend()

df_NorthwardWindShelf['northward_wind'].plot(ax=ax4, label='Shelf mooring', color='blue')
df_NorthwardWindOffshore['northward_wind'].plot(ax=ax4, label='Offshore mooring', color='cyan')
ax4.set_xlabel('')
ax4.set_ylabel('Northward wind speed (m/s)')
ax4.legend()

df_RelativeHumidityShelf['relative_humidity'].plot(ax=ax2, label='Shelf mooring', color='blue')
df_RelativeHumidityOffshore['relative_humidity'].plot(ax=ax2, label='Offshore mooring', color='cyan')
ax2.set_xlabel('')
ax2.set_ylabel('Relative humidity (%)')
ax2.legend()

df_ShortWaveFluxShelf['downwelling_shortwave_flux_in_air'].plot(ax=ax5, label='Shelf mooring', color='blue')
df_ShortWaveFluxOffshore['downwelling_shortwave_flux_in_air'].plot(ax=ax5, label='Offshore mooring', color='cyan')
ax5.set_xlabel('')
ax5.set_ylabel('Shortwave irradiance flux (W/m$^2$)')
ax5.legend()

fig.suptitle('Meteorological data')
fig.tight_layout()
```

</details>

## 3.2 The ash-to-bloom connection

***Question: Did ash cause a bloom?***

Answer this question visually by comparing the timing of changes in any two or more of the sea surface parameters we already worked with, especially in conunction with the meteorological time-series. Sea surface data is available from all three moorings. Follow the data retrieval, data processing, and graphing approaches you practiced in 3.1 above.

In [ ]:
# Enter your code here

<details>
  <summary>Click here to reveal code that may work!</summary>

```python
# Instructor's resource: 3.2 - The ash-to-bloom connection

# Access data from inshore mooring
url_3WavelengthFluorometerInshore = e.get_download_url(
    dataset_id = "ooi-ce01issm-rid16-02-flortd000",
    constraints = {
      "time>=": "2020-09-06",
      "time<=": "2020-09-21",
    }
)
url_O2Inshore = e.get_download_url(
    dataset_id = "ooi-ce01issm-rid16-03-dostad000",
    constraints = {
      "time>=": "2020-09-06",
      "time<=": "2020-09-21",
    }
)
url_pCO2Inshore = e.get_download_url(
    dataset_id = "ooi-ce01issm-rid16-05-pco2wb000",
    constraints = {
      "time>=": "2020-09-06",
      "time<=": "2020-09-21",
    }
)
url_NitrateInshore = e.get_download_url(
    dataset_id = "ooi-ce01issm-rid16-07-nutnrb000",
    constraints = {
      "time>=": "2020-09-06",
      "time<=": "2020-09-21",
    }
)

# Access data from shelf mooring
url_3WavelengthFluorometerShelf = e.get_download_url(
    dataset_id = "ooi-ce02shsm-rid27-02-flortd000",
    constraints = {
      "time>=": "2020-09-06",
      "time<=": "2020-09-21",
    }
)
url_O2Shelf = e.get_download_url(
    dataset_id = "ooi-ce02shsm-rid27-04-dostad000",
    constraints = {
      "time>=": "2020-09-06",
      "time<=": "2020-09-21",
    }
)
url_pCO2Shelf = e.get_download_url(
    dataset_id = "ooi-ce02shsm-sbd12-04-pco2aa000",
    constraints = {
      "time>=": "2020-09-06",
      "time<=": "2020-09-21",
    }
)
url_NitrateShelf = e.get_download_url(
    dataset_id = "ooi-ce02shsm-rid26-07-nutnrb000",
    constraints = {
      "time>=": "2020-09-06",
      "time<=": "2020-09-21",
    }
)

# Access data from offshore mooring
url_pCO2AirSeaOffshore = e.get_download_url(
    dataset_id = "ooi-ce04ossm-sbd12-04-pco2aa000",
    constraints = {
      "time>=": "2020-09-06",
      "time<=": "2020-09-21",
    }
)

# Load data from the Inshore Mooring
df_3WavelengthFluorometerInshore = pd.read_csv(url_3WavelengthFluorometerInshore, index_col='time', parse_dates=True, skiprows=[1])
df_O2Inshore = pd.read_csv(url_O2Inshore, index_col='time', parse_dates=True, skiprows=[1])
df_pCO2Inshore = pd.read_csv(url_pCO2Inshore, index_col='time', parse_dates=True, skiprows=[1])
df_NitrateInshore = pd.read_csv(url_NitrateInshore, index_col='time', parse_dates=True, skiprows=[1])

# Load data from the Shelf Mooring
df_3WavelengthFluorometerShelf = pd.read_csv(url_3WavelengthFluorometerShelf, index_col='time', parse_dates=True, skiprows=[1])
df_O2Shelf = pd.read_csv(url_O2Shelf, index_col='time', parse_dates=True, skiprows=[1])
df_pCO2Shelf = pd.read_csv(url_pCO2Shelf, index_col='time', parse_dates=True, skiprows=[1])
df_NitrateShelf = pd.read_csv(url_NitrateShelf, index_col='time', parse_dates=True, skiprows=[1])

# Load data from the Offshore Mooring
df_pCO2AirSeaOffshore = pd.read_csv(url_pCO2AirSeaOffshore, index_col='time', parse_dates=True, skiprows=[1])

# Remove rows with NaN values
df_SeaSurfaceTemperatureInshore = df_3WavelengthFluorometerInshore.dropna(subset='sea_water_temperature')
df_OpticalBackscatterInshore = df_3WavelengthFluorometerInshore.dropna(subset='flubsct')
df_CDOMInshore = df_3WavelengthFluorometerInshore.dropna(subset='cdomflo')
df_ChlaInshore = df_3WavelengthFluorometerInshore.dropna(subset='mass_concentration_of_chlorophyll_a_in_sea_water')
df_O2Inshore = df_O2Inshore.dropna(subset='mole_concentration_of_dissolved_molecular_oxygen_in_sea_water')
df_pCO2Inshore = df_pCO2Inshore.dropna(subset='partial_pressure_of_carbon_dioxide_in_sea_water')
df_NitrateseaInshore = df_NitrateInshore.dropna(subset='mole_concentration_of_nitrate_in_sea_water_suna')
df_SeaSurfaceTemperatureShelf = df_3WavelengthFluorometerShelf.dropna(subset='sea_water_temperature')
df_OpticalBackscatterShelf = df_3WavelengthFluorometerShelf.dropna(subset='flubsct')
df_CDOMShelf = df_3WavelengthFluorometerShelf.dropna(subset='cdomflo')
df_ChlaShelf = df_3WavelengthFluorometerShelf.dropna(subset='mass_concentration_of_chlorophyll_a_in_sea_water')
df_O2Shelf = df_O2Shelf.dropna(subset='mole_concentration_of_dissolved_molecular_oxygen_in_sea_water')
df_pCO2Shelf = df_pCO2Shelf.dropna(subset='surface_partial_pressure_of_carbon_dioxide_in_sea_water')
df_NitrateShelf = df_NitrateShelf.dropna(subset='mole_concentration_of_nitrate_in_sea_water_suna')
df_pCO2Offshore = df_pCO2AirSeaOffshore.dropna(subset='surface_partial_pressure_of_carbon_dioxide_in_sea_water')

# Keep only nitrate concentration values that are above 6 μmol/L
df_NitrateInshore_filtered = df_NitrateInshore[df_NitrateInshore['mole_concentration_of_nitrate_in_sea_water_suna'] > 6]
df_NitrateShelf_filtered = df_NitrateShelf[
    (df_NitrateShelf['mole_concentration_of_nitrate_in_sea_water_suna'] > 6) &
    (df_NitrateShelf['mole_concentration_of_nitrate_in_sea_water_suna'] < 30)
]

# Plot ash deposition data - ENTER the following in a separate code box
fig,(ax1,ax2,ax3) = plt.subplots(3,1, sharex=True, figsize=(10,10))

df_SeaSurfaceTemperatureInshore['sea_water_temperature'].plot(ax=ax1, label='Inshore mooring', color='navy')
df_SeaSurfaceTemperatureShelf['sea_water_temperature'].plot(ax=ax1, label='Shelf mooring', color='blue')
df_SeaSurfaceTemperatureOffshore['sea_water_temperature'].plot(ax=ax1, label='Offshore mooring', color='cyan')
ax1.set_xlabel('')
ax1.set_ylabel('Sea surface temperature ($^o$C)')
ax1.legend()

df_CDOMInshore['cdomflo'].plot(ax=ax3, label='Inshore mooring', color='navy')
df_CDOMShelf['cdomflo'].plot(ax=ax3, label='Shelf mooring', color='blue')
df_CDOMOffshore['cdomflo'].plot(ax=ax3, label='Offshore mooring', color='cyan')
ax3.set_xlabel('')
ax3.set_ylabel('CDOM (ppb)')
ax3.legend()

df_OpticalBackscatterInshore['flubsct'].plot(ax=ax2, label='Inshore mooring', color='navy')
df_OpticalBackscatterShelf['flubsct'].plot(ax=ax2, label='Shelf mooring', color='blue')
df_OpticalBackscatterOffshore['flubsct'].plot(ax=ax2, label='Offshore mooring', color='cyan')
ax2.set_xlabel('')
ax2.set_ylabel('Optical backscatter (1/m)')
ax2.legend()

fig.suptitle('Ash deposition data')
fig.tight_layout()

# Plot algal bloom data - ENTER the following in a separate code box
fig,(ax1,ax2,ax3,ax4) = plt.subplots(4,1, sharex=True, figsize=(10,10))

df_O2Inshore['mole_concentration_of_dissolved_molecular_oxygen_in_sea_water'].plot(ax=ax2, label='Inshore mooring', color='navy')
df_O2Shelf['mole_concentration_of_dissolved_molecular_oxygen_in_sea_water'].plot(ax=ax2, label='Shelf mooring', color='blue')
df_O2Offshore['mole_concentration_of_dissolved_molecular_oxygen_in_sea_water'].plot(ax=ax2, label='Offshore mooring', color='cyan')
ax2.set_xlabel('')
ax2.set_ylabel('[O$_2$] (μmol/L)')
ax2.legend()

df_pCO2Inshore['partial_pressure_of_carbon_dioxide_in_sea_water'].plot(ax=ax3, label='Inshore mooring', color='navy')
df_pCO2Shelf['surface_partial_pressure_of_carbon_dioxide_in_sea_water'].plot(ax=ax3, label='Shelf mooring', color='blue')
df_pCO2Offshore['surface_partial_pressure_of_carbon_dioxide_in_sea_water'].plot(ax=ax3, label='Offshore mooring', color='cyan')
ax3.set_xlabel('')
ax3.set_ylabel('$p$CO$_2$(μatm)')
ax3.legend()

df_NitrateInshore_filtered['mole_concentration_of_nitrate_in_sea_water_suna'].plot(ax=ax4, label='Inshore mooring', color='navy')
df_NitrateShelf_filtered['mole_concentration_of_nitrate_in_sea_water_suna'].plot(ax=ax4, label='Shelf mooring', color='blue')
df_NitrateOffshore_filtered['mole_concentration_of_nitrate_in_sea_water_suna'].plot(ax=ax4, label='Offshore mooring', color='cyan')
ax4.set_xlabel('')
ax4.set_ylabel('[NO$_3^{-}$] (μmol/L)')
ax4.legend()

df_ChlaInshore['mass_concentration_of_chlorophyll_a_in_sea_water'].plot(ax=ax1, label='Inshore mooring', color='navy')
df_ChlaShelf['mass_concentration_of_chlorophyll_a_in_sea_water'].plot(ax=ax1, label='Shelf mooring', color='blue')
df_ChlaOffshore['mass_concentration_of_chlorophyll_a_in_sea_water'].plot(ax=ax1, label='Offshore mooring', color='cyan')
ax1.set_xlabel('')
ax1.set_ylabel('[Chlorophyll $a$] (μg/L)')
ax1.legend()

fig.suptitle('Algal bloom data')
fig.tight_layout()
```

</details>

## 3.3 Before, during and after

***Question: By how much did key properties change before, during and after the ash arrived at the offshore mooring?***

You can answer this question by:

*   Spliting the data from the three stages: before, during, and after ash arrival
*   Calculating descriptive metrics of the parameter values at these three stages, e.g., average, median, minimum, maximum, etc.
*   Calculating the difference in the metrics above between the three stages
*   Generating box plots of the data in the three stages

In [ ]:
# Enter your code here

## 3.4 Appearances may deceive

***Question: Are optical sensors reliably reporting changes in the values of properties they are supposed to be monitoring or is their record affected by ash settling in the water?***

Many of the biogeochemical sensors we use are optical, i.e., they are recording light properties of the water that act as proxies of the biogeochemical properties themselves. Examples of optical sensors used in this dataset include backscatter, oxygen and chlorophyll concentrations. Ash settling on the water may interfere with their measurements.

To answer this question:

*   Examine the time-series of these properties and note similarities or differences
*   Compare their values visually by using the graphing approaches proposed in this section but also using scatter plots

In [ ]:
# Enter your code here

<details>
  <summary>Click here to reveal code that may work!</summary>

```python
# Instructor's resource: 3.4 - Example of comparison of oxygen (optical sensor) and carbon dioxide (non-optical sensor)

# Combine the relevant columns from the two relevant dataframes (oxygen and carbon dioxide) and drop rows with NaN values
df_InshoreO2vspCO2 = pd.concat([df_O2Inshore['mole_concentration_of_dissolved_molecular_oxygen_in_sea_water'], df_pCO2Inshore['partial_pressure_of_carbon_dioxide_in_sea_water']], axis=1).dropna()
df_ShelfO2vspCO2 = pd.concat([df_O2Shelf['mole_concentration_of_dissolved_molecular_oxygen_in_sea_water'], df_pCO2Shelf['surface_partial_pressure_of_carbon_dioxide_in_sea_water']], axis=1).dropna()
df_OffshoreO2vspCO2 = pd.concat([df_O2Offshore['mole_concentration_of_dissolved_molecular_oxygen_in_sea_water'], df_pCO2Offshore['surface_partial_pressure_of_carbon_dioxide_in_sea_water']], axis=1).dropna()

# Plot oxygen against carbon dioxide concentrations for each of the three moorings on the same graph
plt.figure()
plt.scatter(df_InshoreO2vspCO2['mole_concentration_of_dissolved_molecular_oxygen_in_sea_water'], df_InshoreO2vspCO2['partial_pressure_of_carbon_dioxide_in_sea_water'], label='Inshore mooring', color='navy')
plt.scatter(df_ShelfO2vspCO2['mole_concentration_of_dissolved_molecular_oxygen_in_sea_water'], df_ShelfO2vspCO2['surface_partial_pressure_of_carbon_dioxide_in_sea_water'], label='Shelf mooring', color='blue')
plt.scatter(df_OffshoreO2vspCO2['mole_concentration_of_dissolved_molecular_oxygen_in_sea_water'], df_OffshoreO2vspCO2['surface_partial_pressure_of_carbon_dioxide_in_sea_water'], label='Offshore mooring', color='cyan')
plt.xlabel('[O$_2$] (μmol/L)')
plt.ylabel('$p$CO$_2$ (μatm)')
plt.legend()
plt.grid(True)
plt.show()

# Important data literacy lesson:
# Question: Why is the inshore mooring data series not visible?
# The two sensors didn't report values at the same exact times, so when we remove NaN values from the combined table, all rows get deleted!

```

</details>